In [1]:
# CLEANUP OLD FILES
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
processed_dir = PROJECT_ROOT / "outputs" / "processed"

# Files to delete
old_files = [
    "shap_beeswarm_data.csv",
    "shap_importance.csv", 
    "shap_values.csv",
]

plots_dir = PROJECT_ROOT / "outputs" / "plots"
old_plot_files = [
    plots_dir / "xai_beeswarm.html",
    plots_dir / "xai_elevation_dependence.png",
    plots_dir / "xai_global_importance.png",
]

print("🗑️  Cleaning up deprecated files...\n")

for f in old_files:
    file_path = processed_dir / f
    if file_path.exists():
        file_path.unlink()
        print(f"✅ Deleted: {file_path}")
    else:
        print(f" Not found: {file_path}")

for f in old_plot_files:
    if f.exists():
        f.unlink()
        print(f"✅ Deleted: {f}")
    else:
        print(f"⊘ Not found: {f}")

print("\n✨ Cleanup complete!")

🗑️  Cleaning up deprecated files...

 Not found: d:\my_project\outputs\processed\shap_beeswarm_data.csv
 Not found: d:\my_project\outputs\processed\shap_importance.csv
 Not found: d:\my_project\outputs\processed\shap_values.csv
⊘ Not found: d:\my_project\outputs\plots\xai_beeswarm.html
⊘ Not found: d:\my_project\outputs\plots\xai_elevation_dependence.png
⊘ Not found: d:\my_project\outputs\plots\xai_global_importance.png

✨ Cleanup complete!


In [2]:
%load_ext autoreload
%autoreload 2

import sys
import joblib
import warnings
from pathlib import Path
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore', category=UserWarning)

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.xai_explainer import SHAPExplainer

model_dir = PROJECT_ROOT / "outputs" / "processed"
ml_parquet = PROJECT_ROOT / "outputs" / "processed" / "sarawak_TCA_fused.parquet"

print("✅ Setup complete.")

c:\Users\User\miniconda3\envs\hydro-ml\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.17) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✅ Setup complete.


In [3]:
# CELL 2: MULTI-MODEL XAI PROCESSING (FIXED VERSION)
import joblib
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# Make sure we're using the right path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.xai_explainer import SHAPExplainer

model_dir = PROJECT_ROOT / "outputs" / "processed"
ml_parquet = PROJECT_ROOT / "outputs" / "processed" / "sarawak_ML_corrected.parquet"

# Find all saved model files
model_files = list(model_dir.glob("model_*.pkl"))

if not model_files:
    raise FileNotFoundError("❌ No 'model_*.pkl' files found. Please run Cell 2 in 05_ml_bias_correction.ipynb first.")

print(f"🔍 Found {len(model_files)} model(s) to analyze.\n")

for model_path in model_files:
    # Extract clean model name
    model_name = model_path.stem.replace("model_", "").replace("_", " ")
    safe_name = model_path.stem.replace("model_", "")
    
    print(f" Processing xAI for: {model_name}")
    
    try:
        # Initialize explainer
        explainer = SHAPExplainer(str(ml_parquet), str(model_dir))
        
        # Load THIS specific model (don't overwrite the class!)
        current_model = joblib.load(model_path)
        explainer.model = current_model
        explainer.model_type = type(current_model).__name__
        
        print(f"   Data loaded. {len(explainer.df_valid)} valid samples available.")
        
        # Compute SHAP (Use smaller sample for non-tree models)
        if explainer.model_type in ['RandomForestRegressor', 'GradientBoostingRegressor', 'ExtraTreesRegressor']:
            sample_size = 2000
        else:
            sample_size = 500 
            
        print(f"   Computing SHAP for {explainer.model_type} (n={sample_size})...")
        shap_values, X_sample, original_sample_df = explainer.compute_shap_values(sample_size=sample_size)
        
        # Get dataframes
        importance_df = explainer.get_global_importance(shap_values)
        beeswarm_df = explainer.get_beeswarm_data(shap_values, X_sample, original_sample_df)
        
        # Save with model-specific names
        importance_df.to_csv(model_dir / f"shap_importance_{safe_name}.csv", index=False)
        beeswarm_df.to_csv(model_dir / f"shap_beeswarm_data_{safe_name}.csv", index=False)
        
        print(f"   Successfully saved xAI data for {model_name}\n")
        
    except Exception as e:
        print(f"   Error: {e}")
        print(f"   Failed to process {model_name}: {e}\n")
        import traceback
        traceback.print_exc()

print("🎉 All xAI processing complete!")

🔍 Found 4 model(s) to analyze.

 Processing xAI for: Gradient Boosting
 Loaded model type: GradientBoostingRegressor
✅ Data loaded. 107949 valid samples available.
   Data loaded. 107949 valid samples available.
   Computing SHAP for GradientBoostingRegressor (n=2000)...
🧮 Computing SHAP for GradientBoostingRegressor (n=2000)...
   → Using TreeExplainer (optimized for tree-based models)
✅ SHAP values computed successfully.
   Successfully saved xAI data for Gradient Boosting

 Processing xAI for: Linear Regression
 Loaded model type: GradientBoostingRegressor
✅ Data loaded. 107949 valid samples available.
   Data loaded. 107949 valid samples available.
   Computing SHAP for LinearRegression (n=500)...
🧮 Computing SHAP for LinearRegression (n=500)...
   → Using KernelExplainer (fallback for LinearRegression)


  0%|          | 0/500 [00:00<?, ?it/s]

✅ SHAP values computed successfully.
   Successfully saved xAI data for Linear Regression

 Processing xAI for: Neural Network MLP
 Loaded model type: GradientBoostingRegressor
✅ Data loaded. 107949 valid samples available.
   Data loaded. 107949 valid samples available.
   Computing SHAP for MLPRegressor (n=500)...
🧮 Computing SHAP for MLPRegressor (n=500)...
   → Using PermutationExplainer (for MLP neural network)
   ️ This may take a few minutes...
   → PermutationExplainer failed: 'MLPRegressor' object is not callable
   → Falling back to KernelExplainer


  0%|          | 0/500 [00:00<?, ?it/s]

✅ SHAP values computed successfully.
   Successfully saved xAI data for Neural Network MLP

 Processing xAI for: Random Forest
 Loaded model type: GradientBoostingRegressor
✅ Data loaded. 107949 valid samples available.
   Data loaded. 107949 valid samples available.
   Computing SHAP for RandomForestRegressor (n=2000)...
🧮 Computing SHAP for RandomForestRegressor (n=2000)...
   → Using TreeExplainer (optimized for tree-based models)
✅ SHAP values computed successfully.
   Successfully saved xAI data for Random Forest

🎉 All xAI processing complete!
